In [1]:
import pandas as pd
import numpy as np

In [29]:
weighted_returns_df = pd.read_csv("data/outputs/baseline_portfolio",
                                  index_col = 0, parse_dates = [1])
daily_df_annually = pd.read_csv("data/outputs/annually_reblanced_portfolio",
                                index_col = 0, parse_dates = [1])
daily_df_quarterly = pd.read_csv("data/outputs/quarterly_reblanced_portfolio",
                                 index_col = 0, parse_dates = [1])
daily_df_quarterly_band = pd.read_csv("data/outputs/quarterly_reblanced_byband_portfolio", 
                                      index_col = 0, parse_dates = [1])
annual_weights_frame = pd.read_csv("data/outputs/annually_rebalanced_weights",
                                index_col = 0)
quarterly_weights_frame= pd.read_csv("data/outputs/quarterly_rebalanced_weights",
                                index_col = 0)
band_weights_frame = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights",
                                index_col = 0)

In [30]:
daily_df_annually['Date']

0      2022-01-03
1      2022-01-04
2      2022-01-05
3      2022-01-06
4      2022-01-07
          ...    
1126   2026-07-01
1127   2026-07-02
1128   2026-07-06
1129   2026-07-07
1130   2026-07-08
Name: Date, Length: 1131, dtype: datetime64[us]

In [32]:
## Portfolio Drift ##

def drift_evaluation(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]

    difference = np.array(ending_weights.drop(columns=['Date','weight_type'])) - np.array(starting_weights.drop(columns=['Date','weight_type']))

    drift_list = []

    for period in range(len(difference)):
        start_date = starting_weights['Date'].iloc[period]
        end_date = ending_weights['Date'].iloc[period]

        drift = abs(difference[period]).sum()
        drift_list.append({
            "Start": start_date,
            "End" : end_date,
            "Drift": drift
            })

    return pd.DataFrame(drift_list)

In [33]:
## Trades ##
def calculate_trades(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]
    starting_weights_next = starting_weights.drop(starting_weights.index[0])
    ending_weights_current = ending_weights.drop(ending_weights.index[-1])
    start = starting_weights_next.drop(columns=['Date','weight_type'])
    end = ending_weights_current.drop(columns=['Date','weight_type'])
    trades = np.array(start) - np.array(end)
    trades_df = pd.DataFrame(trades, index = starting_weights_next['Date'], columns = start.columns)
    return trades_df

annual_trades_df = calculate_trades(annual_weights_frame)
annual_trades_df

,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ
Date,,,,,,,,,,,,,,
2023-01-01,-0.011494,0.000000e+00,-8.960803e-03,0.150000,1.500000e-01,0.150000,0.000000e+00,-0.041728,0.000000e+00,-0.101925,-1.276777e-01,-0.081914,0.060631,-1.369304e-01
2024-01-01,0.010076,2.884593e-17,-6.754119e-03,0.000757,-1.547203e-01,0.005111,2.259489e-17,0.150000,1.500000e-01,-0.005700,5.181237e-16,-0.034013,-0.114756,4.465968e-17
2025-01-01,0.016766,1.500000e-01,-1.360986e-01,-0.016569,-8.730215e-17,0.017359,1.500000e-01,-0.115151,-1.608208e-01,0.095253,-4.176816e-16,-0.000739,0.000000,-4.103932e-17
2026-01-01,0.020659,-1.166472e-02,5.308254e-16,-0.047497,1.242124e-16,0.019810,-1.473898e-01,0.001424,1.257812e-16,0.007956,4.662246e-16,0.006702,0.150000,0.000000e+00


In [83]:
## Turnover ## 
def calculate_turnover(df):
    turnover_list = []
    for row in range(len(df.index)):
        turnover = abs(df.iloc[row]).sum(axis = 0) * 0.5
        turnover_list.append({
            'Trade Date':df.index[row],
            'turnover': turnover
            })
    return pd.DataFrame(turnover_list)

annual_portfolio_turnover = calculate_turnover(annual_trades_df)
annual_portfolio_turnover

,Trade Date,turnover
0,2023-01-01,0.510631
1,2024-01-01,0.315944
2,2025-01-01,0.429378
3,2026-01-01,0.206552


np.float64(0.3159436679283914)

In [103]:
## Transaction Cost Analysis ##
def transaction_cost_analysis(df, turover_df, bp):

    cost_list = []

    transcation_cost = bp / 10000 

    trade_days = turover_df['Trade Date']

    portfolio_value = pd.Series(df.loc[df["Rebalanced"] == 1, "portfolio_value"].shift().drop(index = 0))

    turnover = pd.Series(turover_df['turnover'])

    for row in range(len(portfolio_value)):
        cost = portfolio_value[row] * turnover[row] * transcation_cost
        cost_list.append(cost)

    return pd.Series(cost_list), portfolio_value, turnover

transaction_cost_analysis(
    df = daily_df_annually,
    turover_df=annual_portfolio_turnover,
    bp = 10
)

KeyError: 0

## Weight Stability ## 
For each rebalance:

Compute turnover
T
Observe portfolio value before the rebalance
V
Assume transaction cost
c
Cost paid
Cost=V×T×c
New portfolio value
V
after
	​

=V−Cost

Holding Period Analysis

Questions

Average holding period

How often is each ETF traded?

Average position age

Number of consecutive quarters held